# Statistical Analysis and Portfolio Optimization

## Overview

This notebook performs statistical analysis on the collected financial data and implements the Markowitz Mean-Variance portfolio optimization algorithm. We analyze asset returns, compute risk metrics, and determine optimal portfolio weights that maximize returns for a given level of risk.

### Key Objectives:
- Load and validate financial time series data
- Compute statistical measures (mean returns, volatilities, correlations)
- Build the covariance matrix for risk assessment
- Formulate and solve the portfolio optimization problem
- Analyze optimal asset allocation

### Mathematical Foundation:
The portfolio optimization problem minimizes variance subject to return and budget constraints:

$$\min_{w} \frac{1}{2} w^T \Sigma w$$
$$\text{subject to:} \quad w^T \mu \geq r_{\text{target}}, \quad w^T 1 = 1, \quad w \geq 0$$

Where:
- $w$: Portfolio weights
- $\Sigma$: Covariance matrix
- $\mu$: Expected returns vector
- $r_{\text{target}}$: Target return

In [ ]:
# Import necessary libraries for analysis and optimization
import numpy as np        # Numerical computations and array operations
import pandas as pd       # Data manipulation and analysis
import cvxpy as cp        # Convex optimization library for portfolio optimization

In [ ]:
# Load summary data (for reference)
df = pd.read_csv("Data.csv")
print("Summary Statistics from Data Collection:")
print(df)

# Extract expected returns from summary data
returns = df.iloc[0, 1:]  # First row contains returns
print("\nExpected Returns:")
print(returns)

## Covariance Matrix Calculation

The summary statistics provide point estimates but cannot generate a proper covariance matrix. For portfolio optimization, we need the covariance between assets, which requires time series data showing how returns co-move over time.

Using the full returns dataset from `returns_data.csv`, we can compute:
- **Covariance Matrix**: Measures joint variability between assets
- **Correlation Matrix**: Standardized measure of relationships
- **Annualized Statistics**: Convert daily metrics to annual scale

In [ ]:
# Load the complete returns time series data
returns_df = pd.read_csv("returns_data.csv", index_col=0)  # Set dates as index

# Compute the covariance matrix and annualize (252 trading days)
cov_matrix = returns_df.cov() * 252
print("Annualized Covariance Matrix (10x10):")
print(cov_matrix.round(6))  # Round for readability

# Compute annualized expected returns and volatilities
mean_returns = returns_df.mean() * 252  # Annualized mean returns
std_devs = returns_df.std() * (252 ** 0.5)  # Annualized standard deviations

print("\nAnnualized Mean Returns:")
print(mean_returns.round(4))
print("\nAnnualized Standard Deviations:")
print(std_devs.round(4))

## Statistical Analysis Summary

From the time series data, we have computed:
- **Covariance Matrix**: Essential for measuring portfolio risk
- **Expected Returns**: Annualized mean returns for each asset
- **Volatilities**: Annualized standard deviations

These statistics form the foundation for portfolio optimization, where we seek to minimize risk (variance) while achieving a target return level.

## Portfolio Optimization Setup

Now we implement the core optimization algorithm. Using CVXPY, we formulate the Markowitz portfolio optimization as a quadratic programming problem.

### Problem Formulation:
- **Decision Variables**: Portfolio weights $w_i$ for each asset
- **Objective**: Minimize portfolio variance $\frac{1}{2} w^T \Sigma w$
- **Constraints**: Budget, return target, and non-negativity

Defining the decision variable: 

In [ ]:
# Define decision variables: portfolio weights for 10 assets
n = 10  # Number of assets
w = cp.Variable(n)  # Weight vector w = [w1, w2, ..., w10]
print("Portfolio weights variable:", w)

## Objective Function

The objective is to minimize portfolio variance (risk):

$$\min_{w} \quad \frac{1}{2} w^T \Sigma w$$

This quadratic form represents the portfolio's total risk, where $\Sigma$ is the covariance matrix capturing both individual asset volatilities and correlations between assets.

In [ ]:
# Define the objective function: minimize portfolio variance
obj = w.T @ cov_matrix @ w  # Quadratic form: w^T * Σ * w
obj_func = cp.Minimize(obj)
print("Objective function:", obj_func)

## Constraints

The optimization is subject to three key constraints:

1. **Budget Constraint**: All weights must sum to 1 (fully invested)
   $$w_1 + w_2 + \dots + w_{10} = 1$$

2. **Return Constraint**: Portfolio must achieve minimum expected return
   $$\mu^T w \geq r_{\text{target}}$$

3. **Non-negativity**: No short selling allowed
   $$w_i \geq 0 \quad \forall i$$

Obtaining the data on max return to set an expected return value:

In [ ]:
# Set target expected return
# First, check the maximum possible return for reference
max_return = mean_returns.max()
print(f"Maximum expected return: {max_return:.4f}")

# Set target return (15% annual return)
e_return = 0.15
print(f"Target expected return: {e_return:.2%}")

Setting the constraints to cvxpy

In [ ]:
# Extract expected returns as numpy array
mu = mean_returns.values

# Define optimization constraints
constraints = [
    cp.sum(w) == 1,           # Budget constraint: weights sum to 1
    mu.T @ w >= e_return,     # Return constraint: achieve target return
    w >= 0                    # Non-negativity: no short selling
]

print("Constraints defined:")
print(f"1. Sum of weights = 1")
print(f"2. Portfolio return >= {e_return:.2%}")
print(f"3. All weights >= 0")

Defining the optimization problem and using the optimization algorithm from cvpxy

In [ ]:
# Formulate the optimization problem
prob = cp.Problem(obj_func, constraints)

# Solve the quadratic programming problem
result = prob.solve()

print(f"Optimization Status: {prob.status}")
print(f"Optimal Portfolio Variance: {prob.value:.6f}")

In [ ]:
# Display optimization results
print(f"Problem Status: {prob.status}")
print(f"Optimal Value (Minimum Variance): {prob.value:.6f}")
print(f"Optimal Weights: {w.value}")
print(f"Sum of Weights: {np.sum(w.value):.6f}")

# Display weights by asset
tickers = ['AAPL', 'MSFT', 'NVDA', 'JPM', 'GS', 'XOM', 'JNJ', 'AMZN', 'PG', 'GLD']
print("\nOptimal Portfolio Allocation:")
for ticker, weight in zip(tickers, w.value):
    if weight > 1e-6:  # Only show significant weights
        print(f"{ticker}: {weight:.4f} ({weight*100:.2f}%)")

In [ ]:
# --- Scenario 2: Minimum-variance portfolio for 10% target return ---
e_return_10 = 0.10
print(f"Target expected return (Scenario 2): {e_return_10:.2%}")

# Reuse the same decision variable, covariance matrix, and expected returns
constraints_10 = [
    cp.sum(w) == 1,
    mu.T @ w >= e_return_10,
    w >= 0
]

prob_10 = cp.Problem(cp.Minimize(w.T @ cov_matrix @ w), constraints_10)
result_10 = prob_10.solve()

print(f"Optimization Status (10%): {prob_10.status}")
print(f"Optimal Portfolio Variance (10%): {prob_10.value:.6f}")

In [ ]:
# Display allocation for the 10% target-return portfolio
weights_10 = np.array(w.value).flatten()
portfolio_return_10 = float(mu @ weights_10)
portfolio_vol_10 = float(np.sqrt(weights_10.T @ cov_matrix.values @ weights_10))

print(f"Problem Status (10%): {prob_10.status}")
print(f"Portfolio Expected Return (10% case): {portfolio_return_10:.4f} ({portfolio_return_10:.2%})")
print(f"Portfolio Volatility (10% case): {portfolio_vol_10:.4f} ({portfolio_vol_10:.2%})")
print(f"Sum of Weights (10% case): {np.sum(weights_10):.6f}")

print("\nOptimal Portfolio Allocation for 10% Target Return:")
for ticker, weight in zip(tickers, weights_10):
    if weight > 1e-6:
        print(f"{ticker}: {weight:.4f} ({weight*100:.2f}%)")

## Comparison Table: 15% vs 10% Target Return

The table below compares optimal weights and portfolio metrics for both target return constraints using the same Markowitz setup.

In [ ]:
# Build a side-by-side comparison table for 15% and 10% target returns
cov_np = cov_matrix.values

# Solve 15% target-return optimization
w_15 = cp.Variable(n)
constraints_15 = [
    cp.sum(w_15) == 1,
    mu.T @ w_15 >= 0.15,
    w_15 >= 0
]
prob_15 = cp.Problem(cp.Minimize(cp.quad_form(w_15, cov_np)), constraints_15)
prob_15.solve()

# Solve 10% target-return optimization
w_10 = cp.Variable(n)
constraints_10_table = [
    cp.sum(w_10) == 1,
    mu.T @ w_10 >= 0.10,
    w_10 >= 0
]
prob_10_table = cp.Problem(cp.Minimize(cp.quad_form(w_10, cov_np)), constraints_10_table)
prob_10_table.solve()

weights_15 = np.array(w_15.value).flatten()
weights_10 = np.array(w_10.value).flatten()

# Asset allocation comparison (in %)
allocation_comparison = pd.DataFrame({
    "Asset": tickers,
    "Weight @ 15% Target (%)": np.round(weights_15 * 100, 3),
    "Weight @ 10% Target (%)": np.round(weights_10 * 100, 3),
    "Difference (10% - 15%)": np.round((weights_10 - weights_15) * 100, 3)
})

# Portfolio-level metric comparison
metrics_comparison = pd.DataFrame({
    "Metric": [
        "Optimization Status",
        "Expected Return (%)",
        "Volatility (%)",
        "Variance",
        "Sum of Weights"
    ],
    "15% Target": [
        prob_15.status,
        round(float(mu @ weights_15) * 100, 3),
        round(float(np.sqrt(weights_15.T @ cov_np @ weights_15)) * 100, 3),
        round(float(weights_15.T @ cov_np @ weights_15), 6),
        round(float(np.sum(weights_15)), 6)
    ],
    "10% Target": [
        prob_10_table.status,
        round(float(mu @ weights_10) * 100, 3),
        round(float(np.sqrt(weights_10.T @ cov_np @ weights_10)) * 100, 3),
        round(float(weights_10.T @ cov_np @ weights_10), 6),
        round(float(np.sum(weights_10)), 6)
    ]
})

print("Asset Allocation Comparison Table:")
print(allocation_comparison.to_string(index=False))

print("\nPortfolio Metrics Comparison Table:")
print(metrics_comparison.to_string(index=False))

## Results Interpretation

The optimization successfully found the minimum-variance portfolio that achieves at least 15% expected return. Note that very small negative weights (near machine precision, ~1e-23) are numerical artifacts and should be treated as zero.

### Key Insights:
- The algorithm allocates capital to achieve the target return with minimal risk
- Assets with negative weights are effectively excluded from the portfolio
- The solution represents a point on the efficient frontier

### Next Steps:
- Analyze the efficient frontier by varying the target return
- Visualize portfolio performance and risk metrics
- Consider additional constraints (transaction costs, sector limits, etc.)